In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

In [ ]:
TASK = Path.cwd().parent
CONFIG = json.loads((TASK.parent/'config.json').read_text())
INPUT, OUTPUT = TASK/'input', TASK/'output'
FIELDS = {field['field_id']:field['label'] for field in CONFIG['fields']}
YEARS = list(range(1, CONFIG['last_publication_year']-CONFIG['first_roster_year']+1))
YEAR_COLUMNS = [str(year) for year in YEARS]
WINDOW = 4
MIN_RISK = 20

In [ ]:
panel = pd.read_csv(INPUT/'secret_aarc_13_field_trajectories.csv', dtype={'dblp_id':'string'})
groups = {'pooled': panel.drop_duplicates('dblp_id'), **dict(tuple(panel.groupby('field_id')))}

In [ ]:
def gini(values):
    ordered = np.sort(values)
    n = len(ordered)
    return 2*np.dot(np.arange(1,n+1),ordered)/(n*ordered.sum())-(n+1)/n if ordered.sum()>0 else np.nan

def rank_correlation(x, y):
    if len(x)>2 and np.ptp(x)>0 and np.ptp(y)>0:
        return stats.spearmanr(x,y).statistic
    return np.nan

def rank_slope(curve, start, reference):
    selected = pd.DataFrame(curve).query('reference == @reference and year >= @start').dropna(subset=['rho'])
    return stats.linregress(selected.year, selected.rho).slope if len(selected)>1 else np.nan

In [ ]:
metrics, qq_points, ranks, dropout, reentry = [], [], [], [], []
for field, group in groups.items():
    q = group[YEAR_COLUMNS].to_numpy(dtype=float).T
    observed = np.isfinite(q)
    complete = q[:, observed.all(axis=0)]

    # draft cutoff: at least five years observed, total the last four
    recent = observed & (observed[::-1].cumsum(axis=0)[::-1]<=WINDOW)
    totals = np.where(recent,q,0).sum(axis=0)[observed.sum(axis=0)>WINDOW]
    positive = totals[totals>0]
    qq_r = np.nan
    if len(positive)>1 and np.ptp(positive)>0:
        log_totals = np.log(positive)
        theoretical, ordered = stats.probplot(log_totals, fit=False)
        qq_r = stats.probplot(log_totals)[1][2]
        standardized = (ordered-log_totals.mean())/log_totals.std(ddof=1)
        qq_points.extend({'field_id':field,'theoretical':x,'standardized_log_total':y}
                         for x,y in zip(theoretical,standardized))

    curve = []
    for j,year in enumerate(YEARS):
        references = [('year_1',0), *[(f'lag_{lag}',j-lag) for lag in (3,5) if j>=lag]]
        curve.extend({'field_id':field,'year':year,'reference':name,'n_complete':complete.shape[1],
                      'rho':rank_correlation(complete[i],complete[j])} for name,i in references)
    ranks.extend(curve)

    valid = observed[:-1] & observed[1:]
    active = valid & (q[:-1]>0)
    current, next_output = q[:-1][active], q[1:][active]
    if len(current):
        interior = np.unique(np.quantile(current,np.linspace(0,1,11)))[1:-1]
        bins = pd.cut(current, [-np.inf,*interior,np.inf], labels=False)
        binned = (pd.DataFrame({'bin':bins+1,'current':current,'dropout':next_output==0})
                  .groupby('bin').agg(n_at_risk=('current','size'),mean_current=('current','mean'),
                                     n_dropout=('dropout','sum'),probability=('dropout','mean')).reset_index())
        dropout.extend(binned.assign(field_id=field).to_dict('records'))

    duration = np.zeros(q.shape[1],dtype=int)
    risk_rows = []
    for j in range(len(YEARS)-1):
        duration = np.where(q[j]==0,duration+1,0)
        inactive = valid[j] & (duration>0)
        risk_rows.append(pd.DataFrame({'duration':duration[inactive],'restart':q[j+1,inactive]>0}))
    hazard = (pd.concat(risk_rows).groupby('duration').restart.agg(n_at_risk='size',n_restart='sum')
              .reset_index())
    hazard['probability'] = hazard.n_restart/hazard.n_at_risk
    hazard['paper_support'] = hazard.n_at_risk.ge(MIN_RISK)
    reentry.extend(hazard.assign(field_id=field).to_dict('records'))

    fixed_final = next(row['rho'] for row in curve if row['reference']=='year_1' and row['year']==YEARS[-1])
    lag5_final = next(row['rho'] for row in curve if row['reference']=='lag_5' and row['year']==YEARS[-1])
    metrics.append({'field_id':field,'label':FIELDS.get(field,'Pooled'),
                    'n_scholars':len(group),'n_four_year_totals':len(totals),'n_positive_totals':len(positive),
                    'qq_r':qq_r,'qq_r2':qq_r**2,'median_total':np.median(totals) if len(totals) else np.nan,
                    'mean_total':totals.mean() if len(totals) else np.nan,'sd_total':totals.std() if len(totals) else np.nan,
                    'gini_total':gini(totals),'n_complete':complete.shape[1],
                    'rank_year_1_final':fixed_final,'rank_year_1_slope':rank_slope(curve,2,'year_1'),
                    'rank_lag_5_final':lag5_final,'rank_lag_5_slope':rank_slope(curve,6,'lag_5')})

In [ ]:
summary = (pd.DataFrame(metrics).set_index('field_id').reindex(['pooled',*FIELDS]).reset_index())
summary['label'] = summary.field_id.map({'pooled':'Pooled',**FIELDS})
count_columns = ['n_scholars','n_four_year_totals','n_positive_totals','n_complete']
summary[count_columns] = summary[count_columns].fillna(0).astype(int)
summary.to_csv(OUTPUT/'empirical_summary.csv', index=False)
for name, rows, columns in [
    ('qq_points',qq_points,['field_id','theoretical','standardized_log_total']),
    ('rank_persistence',ranks,['field_id','year','reference','n_complete','rho']),
    ('dropout',dropout,['field_id','bin','n_at_risk','mean_current','n_dropout','probability']),
    ('reentry',reentry,['field_id','duration','n_at_risk','n_restart','probability','paper_support']),
]:
    pd.DataFrame(rows,columns=columns).to_csv(OUTPUT/f'{name}.csv',index=False)
display(summary)

In [ ]:
points = summary.loc[summary.field_id.ne('pooled')].dropna(subset=['qq_r']).sort_values('qq_r')
if not points.empty:
    sns.set_theme(style='ticks',context='paper',
                  rc={'font.size':8,'axes.labelsize':8,'xtick.labelsize':7,'ytick.labelsize':7,
                      'pdf.fonttype':42,'savefig.dpi':400})
    fig, ax = plt.subplots(figsize=(3.35,2.75),layout='constrained')
    sns.scatterplot(x=points.qq_r,y=list(range(len(points))),ax=ax,legend=False,
                    color=sns.color_palette('colorblind')[2],s=23,linewidth=0)
    ax.set_yticks(range(len(points)),labels=points.label)
    ax.yaxis.tick_right()
    ax.tick_params(axis='y',length=0,pad=5)
    ax.grid(axis='x',color='.9',linewidth=.6)
    ax.set(xlabel=r'Lognormal Q–Q correlation, $r$',ylabel='',
           xlim=(min(.99,max(0,points.qq_r.min()-.006)),1.002))
    sns.despine(ax=ax,left=True)
    for extension in ['pdf','png']:
        fig.savefig(OUTPUT/f'pnas_aarc_field_lognormality.{extension}',facecolor='white')
    plt.close(fig)
else:
    for extension in ['pdf','png']:
        (OUTPUT/f'pnas_aarc_field_lognormality.{extension}').unlink(missing_ok=True)